# ACV-LQN - Object Detection with YOLO, DETR, Faster R-CNN - TH03

## Phát hiện đối tượng trên bộ dữ liệu quân cờ

**Nhóm:** Myopia

| MSHV | Họ tên |
|---|---|
| 25C11067 | Nguyễn Thái Thông |
| 25C11035 | Trần Hạ Khánh Duy |



## Mục lục

- [0. Cài đặt thư viện](#1-cài-đặt-thư-viện)
- [1. Clone repo và checkout nhánh](#2-clone-repo-và-checkout-nhánh)
- [2. Tải dataset từ Google Drive](#3-tải-dataset-từ-google-drive)
- [3. Kiểm tra môi trường](#4-kiểm-tra-môi-trường)
- [4. Chuẩn bị dữ liệu](#5-chuẩn-bị-dữ-liệu)
- [5. Sinh hình minh chứng](#6-sinh-hình-minh-chứng)
- [6. Fine-tune YOLO](#7-fine-tune-yolo)
- [7. Fine-tune DETR](#8-fine-tune-detr)
- [8. Fine-tune Faster R-CNN](#9-fine-tune-faster-r-cnn)
- [9 Kết quả](#10-kết-quả)
- [10. Kết luận](#11-kết-luận)


## 0. Cài đặt thư viện


In [ ]:
%pip install -q -r requirements.txt


## 2. Clone repo và checkout nhánh

Mục này dùng để khai báo mã nguồn thực nghiệm. Điền trực tiếp URL repository, tên nhánh cần sử dụng và thư mục làm việc trên Kaggle trước khi chạy các bước tiếp theo.


In [ ]:
from pathlib import Path

# Điền URL repository dùng cho thí nghiệm.
REPO_URL = 'https://github.com/your-account/your-repository.git'
# Điền tên nhánh chứa mã nguồn cần sử dụng.
BRANCH = 'main'
# Thư mục làm việc trên Kaggle.
REPO_DIR = Path('/kaggle/working/th03_repository')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout {BRANCH}
!git status --short --branch


## 3. Tải dataset từ Google Drive

Dataset được lưu dưới dạng file `.zip` trên Google Drive.


In [ ]:
import zipfile

import gdown

DATASET_FILE_ID = 'YOUR_GOOGLE_DRIVE_FILE_ID'
DATA_DIR = Path('/kaggle/working/data')
ARCHIVE_PATH = DATA_DIR / '416x416_aug_chess_pieces.zip'
DATASET_DIR = DATA_DIR / '416x416_aug_chess_pieces'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if not ARCHIVE_PATH.exists():
    gdown.download(id=DATASET_FILE_ID, output=str(ARCHIVE_PATH), quiet=False)

if not DATASET_DIR.exists() and ARCHIVE_PATH.suffix == '.zip':
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
        zf.extractall(DATA_DIR)

dataset = DATASET_DIR.resolve()
print('Dataset:', dataset)
print('Dataset exists:', dataset.exists())


## 4. Kiểm tra môi trường


In [ ]:
from pathlib import Path
import torch

print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Dataset root:', dataset, dataset.exists())


## 5. Chuẩn bị dữ liệu


In [ ]:
!python source/object_detection.py prepare --dataset {dataset}


## 6. Sinh hình minh chứng


In [ ]:
!python source/object_detection.py figures --dataset {dataset}


In [ ]:
from IPython.display import Image, display

display(Image(filename='doc/figures/sample_ground_truth.png'))
display(Image(filename='doc/figures/split_distribution.png'))
display(Image(filename='doc/figures/class_distribution.png'))


## 7. Fine-tune YOLO


In [ ]:
!python source/object_detection.py yolo --dataset {dataset} --epochs 50 --batch-size 8 --imgsz 416


## 8. Fine-tune DETR


In [ ]:
!python source/object_detection.py detr --dataset {dataset} --epochs 20 --batch-size 2


## 9. Fine-tune Faster R-CNN


In [ ]:
!python source/object_detection.py fasterrcnn --dataset {dataset} --epochs 20 --batch-size 2


## 10. Kết quả

Phần này hiển thị ảnh ground-truth và ảnh prediction sau khi fine-tune YOLO, DETR và Faster R-CNN.


In [ ]:
from pathlib import Path
from IPython.display import Image, display

results_dir = Path('doc/figures')
for image_path in sorted(results_dir.glob('*.png')):
    display(Image(filename=str(image_path)))


## 11. Kết luận

Bài thực hành đã xây dựng pipeline phát hiện đối tượng trên bộ dữ liệu Chess Pieces với ba hướng tiếp cận gồm YOLO, DETR và Faster R-CNN. Dữ liệu đã được chuẩn hóa về các định dạng phù hợp cho từng mô hình, đồng thời các hình minh chứng và ảnh dự đoán sau huấn luyện cũng đã được xuất ra để phục vụ phân tích kết quả.

Trong ba mô hình được khảo sát, YOLO có ưu điểm về tốc độ huấn luyện và suy luận, phù hợp cho các thực nghiệm cần triển khai nhanh. DETR đại diện cho hướng tiếp cận dựa trên transformer, có pipeline dự đoán hiện đại và ít phụ thuộc vào các bước hậu xử lý thủ công hơn. Faster R-CNN là mô hình hai giai đoạn có tính ổn định cao, thường cho kết quả đáng tin cậy trên các bài toán phát hiện đối tượng với số lượng lớp không quá lớn.

Ở bản hiện tại, toàn bộ quy trình từ chuẩn bị dữ liệu, huấn luyện mô hình đến sinh ảnh minh chứng đã được thiết lập đầy đủ. Các kết quả định lượng và hình ảnh dự đoán thu được có thể dùng làm cơ sở để so sánh đặc điểm của từng mô hình, đồng thời là nền tảng để tiếp tục tinh chỉnh siêu tham số như số epoch, batch size và ngưỡng confidence trong các lần thực nghiệm tiếp theo.
